In [3]:

import json
import os
import time
import tracemalloc
from pathlib import Path

import pandas as pd
import psutil

from syngps.utils import (
    build_synthgraph_from_json,
    identify_individual_synthesis_routes,
    find_top_n_routes,
)
from syngps.utils.graph_utils import get_leaf_intermediate_substances

### Config
JSON_FOLDER = "../data/graphs/leaves_as_sm"
TOP_N = 5


In [4]:

def benchmark_graph_file(json_file: Path, top_n: int = 5) -> dict:
    """
    Process a single synthesis graph JSON file and return a metrics dict covering
    graph topology, route statistics, wall-clock timing, CPU time, and memory usage.

    Note: identify_individual_synthesis_routes is called twice per file —
    once here explicitly (for candidate/viable counts) and once internally inside
    find_top_n_routes. Both calls are included in the reported times to reflect
    realistic end-to-end cost.
    """
    process = psutil.Process(os.getpid())

    tracemalloc.start()
    cpu_before = process.cpu_times()
    rss_before = process.memory_info().rss

    # ── Load & build ──────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    with open(json_file) as f:
        data = json.load(f)
    synth_graph = build_synthgraph_from_json(data)
    t1 = time.perf_counter()

    n_nodes = synth_graph.synthesis_graph.number_of_nodes()
    n_edges = synth_graph.synthesis_graph.number_of_edges()
    target = synth_graph.target_molecule_node_id

    # ── Leaf intermediate availability ────────────────────────────────────────
    avail_map = {a["inchikey"]: a for a in (data.get("availability") or [])}

    def is_available(ik):
        a = avail_map.get(ik, {})
        return (
            a.get("inventory", {}).get("available", False)
            or a.get("commercial_availability", {}).get("available", False)
        )

    leaf_ims = get_leaf_intermediate_substances(synth_graph.synthesis_graph)
    n_leaf_unavail = sum(
        1 for n in leaf_ims
        if not is_available(synth_graph.synthesis_graph.nodes[n].get("inchikey", ""))
    )

    # ── Route identification (for candidate / viable counts) ──────────────────
    t2 = time.perf_counter()
    viable_raw, candidates_raw, combo_graphs = identify_individual_synthesis_routes(
        synth_graph.synthesis_graph,
        synth_graph.target_molecule_node_id,
    )
    t3 = time.perf_counter()

    # ── Top-N route ranking ───────────────────────────────────────────────────
    t4 = time.perf_counter()
    routes = find_top_n_routes(synth_graph, top_n)
    t5 = time.perf_counter()

    # ── Finalise measurements ─────────────────────────────────────────────────
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    cpu_after = process.cpu_times()
    rss_after = process.memory_info().rss

    return {
        # Identity
        "file": json_file.name,
        "target_molecule": target,
        # Graph topology
        "nodes": n_nodes,
        "edges": n_edges,
        "leaf_ims_total": len(leaf_ims),
        "leaf_ims_available": len(leaf_ims) - n_leaf_unavail,
        "leaf_ims_unavailable": n_leaf_unavail,
        # Route statistics
        "combination_graphs": len(combo_graphs),
        "route_candidates": len(candidates_raw),
        "viable_routes": len(viable_raw),
        "top_n_config": top_n,
        "top_n_returned": len(routes),
        # Timing (seconds)
        "build_time_s": round(t1 - t0, 4),
        "identify_time_s": round(t3 - t2, 4),
        "find_top_n_time_s": round(t5 - t4, 4),
        "total_time_s": round(t5 - t0, 4),
        # CPU time (seconds)
        "cpu_user_s": round(cpu_after.user - cpu_before.user, 4),
        "cpu_system_s": round(cpu_after.system - cpu_before.system, 4),
        # Memory
        "peak_alloc_mb": round(peak_bytes / 1024**2, 3),
        "rss_delta_mb": round((rss_after - rss_before) / 1024**2, 3),
    }


In [5]:

json_path = Path(JSON_FOLDER)
json_files = sorted(json_path.glob("*.json"))
print(f"Found {len(json_files)} JSON files in {json_path.resolve()}\n")

records = []
errors = []

for i, jf in enumerate(json_files, 1):
    try:
        rec = benchmark_graph_file(jf, top_n=TOP_N)
        records.append(rec)
        print(
            f"[{i:>3}/{len(json_files)}] {jf.name:<55} "
            f"{rec['total_time_s']:>6.3f}s  "
            f"nodes={rec['nodes']:>4}  edges={rec['edges']:>4}  "
            f"viable={rec['viable_routes']:>2}  top_n={rec['top_n_returned']:>2}  "
            f"peak={rec['peak_alloc_mb']:>6.2f}MB"
        )
    except Exception as exc:
        errors.append({"file": jf.name, "error": str(exc)})
        print(f"[{i:>3}/{len(json_files)}] ERROR  {jf.name}: {exc}")

print(f"\n✓ Processed {len(records)} files successfully, {len(errors)} errors.")


Found 160 JSON files in /Users/millernar/Development/syngps/revision/data/graphs/leaves_as_sm

[  1/160] AISQMYCMHKTHHN-SSEXGKCCSA-N_response.json                1.124s  nodes=   4  edges=   3  viable= 1  top_n= 1  peak=  7.46MB
[  2/160] ALQUTEKNDPODSS-UHFFFAOYSA-N_response.json                0.011s  nodes=   7  edges=   6  viable= 1  top_n= 1  peak=  0.10MB
[  3/160] ASSZZQOXRJNXNL-NRFANRHFSA-N_response.json                0.015s  nodes=  21  edges=  22  viable= 1  top_n= 1  peak=  0.25MB
[  4/160] AWDRDIZACXEMRA-UHFFFAOYSA-N_response.json                0.011s  nodes=  12  edges=  11  viable= 1  top_n= 1  peak=  0.12MB
[  5/160] AXZKSUNGRDJIFL-UHFFFAOYSA-N_response.json                0.013s  nodes=  27  edges=  36  viable= 1  top_n= 1  peak=  0.16MB
[  6/160] BFIQWRSPSDOJCV-UHFFFAOYSA-N_response.json                0.012s  nodes=  14  edges=  13  viable= 1  top_n= 1  peak=  0.12MB
[  7/160] BHKKSKOHRFHHIN-MRVPVSSYSA-N_response.json                0.016s  nodes=  23  edges=  26  vi

In [6]:

df = pd.DataFrame(records)

# Ensure consistent column ordering
col_order = [
    "file", "target_molecule",
    "nodes", "edges",
    "leaf_ims_total", "leaf_ims_available", "leaf_ims_unavailable",
    "combination_graphs", "route_candidates", "viable_routes",
    "top_n_config", "top_n_returned",
    "build_time_s", "identify_time_s", "find_top_n_time_s", "total_time_s",
    "cpu_user_s", "cpu_system_s",
    "peak_alloc_mb", "rss_delta_mb",
]
df = df[[c for c in col_order if c in df.columns]]

print(f"DataFrame shape: {df.shape}")
df.head(10)


DataFrame shape: (160, 20)


,file,target_molecule,nodes,edges,leaf_ims_total,leaf_ims_available,leaf_ims_unavailable,combination_graphs,route_candidates,viable_routes,top_n_config,top_n_returned,build_time_s,identify_time_s,find_top_n_time_s,total_time_s,cpu_user_s,cpu_system_s,peak_alloc_mb,rss_delta_mb
0,AISQMYCMHKTHHN-SSEXGKCCSA-N_response.json,AISQMYCMHKTHHN-SSEXGKCCSA-N,4,3,0,0,0,1,1,1,5,1,0.0057,0.0055,1.1124,1.1237,0.3995,0.0524,7.464,19.516
1,ALQUTEKNDPODSS-UHFFFAOYSA-N_response.json,ALQUTEKNDPODSS-UHFFFAOYSA-N,7,6,0,0,0,1,1,1,5,1,0.0043,0.0020,0.0046,0.0109,0.0074,0.0002,0.103,0.125
2,ASSZZQOXRJNXNL-NRFANRHFSA-N_response.json,ASSZZQOXRJNXNL-NRFANRHFSA-N,21,22,0,0,0,1,1,1,5,1,0.0050,0.0034,0.0066,0.0150,0.0114,0.0002,0.248,0.281
3,AWDRDIZACXEMRA-UHFFFAOYSA-N_response.json,AWDRDIZACXEMRA-UHFFFAOYSA-N,12,11,0,0,0,1,1,1,5,1,0.0043,0.0024,0.0045,0.0112,0.0076,0.0001,0.121,0.094
4,AXZKSUNGRDJIFL-UHFFFAOYSA-N_response.json,AXZKSUNGRDJIFL-UHFFFAOYSA-N,27,36,0,0,0,1,1,1,5,1,0.0060,0.0025,0.0044,0.0129,0.0085,0.0003,0.161,0.078
5,BFIQWRSPSDOJCV-UHFFFAOYSA-N_response.json,BFIQWRSPSDOJCV-UHFFFAOYSA-N,14,13,0,0,0,1,1,1,5,1,0.0046,0.0026,0.0047,0.0119,0.0080,0.0003,0.125,0.109
6,BHKKSKOHRFHHIN-MRVPVSSYSA-N_response.json,BHKKSKOHRFHHIN-MRVPVSSYSA-N,23,26,0,0,0,1,1,1,5,1,0.0057,0.0040,0.0062,0.0160,0.0116,0.0002,0.263,0.094
7,BIAAFFAHROMCHN-UHFFFAOYSA-N_response.json,BIAAFFAHROMCHN-UHFFFAOYSA-N,22,21,0,0,0,1,1,1,5,1,0.0052,0.0024,0.0049,0.0126,0.0084,0.0003,0.147,0.047
8,BSHFCWWTFQFTPC-UHFFFAOYSA-N_response.json,BSHFCWWTFQFTPC-UHFFFAOYSA-N,17,16,0,0,0,1,1,1,5,1,0.0063,0.0029,0.0063,0.0155,0.0102,0.0001,0.220,0.125
9,BSWAVDINBJTJNU-OAHLLOKOSA-N_response.json,BSWAVDINBJTJNU-OAHLLOKOSA-N,5,4,0,0,0,1,1,1,5,1,0.0045,0.0017,0.0047,0.0109,0.0067,0.0003,0.073,0.000


In [7]:

# Descriptive statistics for all numeric columns
numeric_cols = [
    "nodes", "edges",
    "leaf_ims_total", "leaf_ims_available", "leaf_ims_unavailable",
    "combination_graphs", "route_candidates", "viable_routes", "top_n_returned",
    "build_time_s", "identify_time_s", "find_top_n_time_s", "total_time_s",
    "cpu_user_s", "cpu_system_s",
    "peak_alloc_mb", "rss_delta_mb",
]
df[numeric_cols].describe().round(4)


,nodes,edges,leaf_ims_total,leaf_ims_available,leaf_ims_unavailable,combination_graphs,route_candidates,viable_routes,top_n_returned,build_time_s,identify_time_s,find_top_n_time_s,total_time_s,cpu_user_s,cpu_system_s,peak_alloc_mb,rss_delta_mb
count,160.0000,160.0000,160.0,160.0,160.0,160.0000,160.0000,160.0000,160.0000,160.0000,160.0000,160.0000,160.0000,160.0000,160.0000,160.0000,160.0000
mean,31.3875,43.0625,0.0,0.0,0.0,1.4375,1.4375,1.4062,1.3188,0.0083,0.0065,0.0218,0.0366,0.0254,0.0007,0.3950,0.2757
std,64.0459,110.8368,0.0,0.0,0.0,1.6085,1.6085,1.5471,0.7214,0.0044,0.0244,0.0984,0.1103,0.0742,0.0042,1.0535,1.7823
min,4.0000,3.0000,0.0,0.0,0.0,1.0000,1.0000,1.0000,1.0000,0.0037,0.0014,0.0039,0.0096,0.0058,0.0001,0.0620,0.0000
25%,7.7500,6.7500,0.0,0.0,0.0,1.0000,1.0000,1.0000,1.0000,0.0050,0.0020,0.0044,0.0128,0.0071,0.0001,0.0990,0.0000
50%,17.0000,17.5000,0.0,0.0,0.0,1.0000,1.0000,1.0000,1.0000,0.0079,0.0025,0.0051,0.0158,0.0089,0.0002,0.1625,0.0000
75%,26.0000,28.2500,0.0,0.0,0.0,1.0000,1.0000,1.0000,1.0000,0.0102,0.0038,0.0080,0.0252,0.0154,0.0003,0.2622,0.0310
max,467.0000,831.0000,0.0,0.0,0.0,19.0000,19.0000,19.0000,5.0000,0.0340,0.3050,1.1124,1.1237,0.7203,0.0524,7.7410,19.5160


In [8]:

# ── Timing breakdown: which phase dominates? ──────────────────────────────────
timing_summary = df[["build_time_s", "identify_time_s", "find_top_n_time_s", "total_time_s"]].agg(
    ["mean", "median", "max", "sum"]
).round(4)
print("=== Timing breakdown (seconds) ===")
print(timing_summary.to_string())

# ── Graphs with viable routes vs fully disqualified ───────────────────────────
print("\n=== Route viability summary ===")
viability = df["viable_routes"].apply(lambda v: "has_viable" if v > 0 else "all_disqualified").value_counts()
print(viability.to_string())

# ── Correlation: does graph size predict processing time? ────────────────────
print("\n=== Correlation with total_time_s ===")
corr_cols = ["nodes", "edges", "combination_graphs", "route_candidates", "viable_routes", "total_time_s"]
print(df[corr_cols].corr()[["total_time_s"]].round(4).to_string())

# ── Top 10 slowest graphs ─────────────────────────────────────────────────────
print("\n=== Top 10 slowest graphs ===")
slowest = df.nlargest(10, "total_time_s")[["file", "nodes", "edges", "viable_routes", "total_time_s", "peak_alloc_mb"]]
print(slowest.to_string(index=False))


=== Timing breakdown (seconds) ===
        build_time_s  identify_time_s  find_top_n_time_s  total_time_s
mean          0.0083           0.0065             0.0218        0.0366
median        0.0079           0.0025             0.0051        0.0158
max           0.0340           0.3050             1.1124        1.1237
sum           1.3288           1.0342             3.4851        5.8553

=== Route viability summary ===
viable_routes
has_viable    160

=== Correlation with total_time_s ===
                    total_time_s
nodes                     0.4049
edges                     0.4345
combination_graphs        0.4906
route_candidates          0.4906
viable_routes             0.4882
total_time_s              1.0000

=== Top 10 slowest graphs ===
                                     file  nodes  edges  viable_routes  total_time_s  peak_alloc_mb
AISQMYCMHKTHHN-SSEXGKCCSA-N_response.json      4      3              1        1.1237          7.464
OYGAAIIWGPRZSJ-UHFFFAOYSA-N_response.json   